# AI 기반 반려견 산책로 추천 모델

## 목표

사용자의 현재 위치(`latitude`, `longitude`)를 기준으로 OSM(OpenStreetMap) 보행 데이터를 조회하여
2~3km 범위의 산책 후보 경로를 생성하고, XGBoost 모델로 각 경로의 산책 적합도 점수를 예측한다.

예측 점수가 높은 상위 3~4개의 경로만 선택하여 FastAPI에서 JSON으로 반환하고,
React 메인 페이지에서 추천 산책로 리스트와 Kakao Map 경로를 표시하는 것을 목표로 한다.

### 전체 처리 흐름

`현재 위치 좌표`
→ `OSMnx 보행 그래프 생성`
→ `산책 후보 경로 생성`
→ `중복/유사 경로 제거`
→ `Feature 추출`
→ `XGBoost 점수 예측`
→ `TOP 3~4 선정`
→ `경로 좌표(points) 변환`
→ `FastAPI JSON 반환`

### XGBoost 입력 Feature

- `distance_m` : 산책 경로 총 거리
- `overlap_ratio` : 동일 도로 중복 이용 비율(%)
- `walkway_ratio` : 보행로 비율
- `residential_ratio` : 주거지역 도로 비율
- `major_road_ratio` : 주요 간선도로 비율
- `green_ratio` : 공원/녹지 근접 비율

> 참고: 현재 모델은 소규모 수동 라벨 데이터를 이용한 프로젝트 프로토타입이다.
> 실제 서비스 품질을 높이려면 사용자 평가 또는 관리자 검수 데이터를 추가해 재학습하는 것이 필요하다.

## 1. 환경 설정 및 라이브러리

이 노트북은 OSMnx, NetworkX, GeoPandas, pandas, scikit-learn, XGBoost를 사용한다.

필요 패키지 예시:

```bash
pip install osmnx networkx geopandas shapely pandas numpy scikit-learn xgboost
```

In [1]:
from pathlib import Path
from collections import defaultdict
import math

import numpy as np
import pandas as pd
import networkx as nx
import osmnx as ox

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

print("OSMnx:", ox.__version__)

OSMnx: 2.1.1


## 2. 공통 설정

실시간 추천과 모델 학습에서 사용하는 Feature 컬럼을 하나의 상수로 관리한다.
FastAPI에서도 동일한 순서를 사용해야 한다.

In [2]:
FEATURE_COLUMNS = [
    "distance_m",
    "overlap_ratio",
    "walkway_ratio",
    "residential_ratio",
    "major_road_ratio",
    "green_ratio",
]

# ai-server/notebooks/route_generation_cleaned.ipynb 기준 기본 경로
MODEL_PATH = Path("../model/walk_route_model.json")
TRAINING_DATA_PATH = Path("../data/route_training_data.csv")

# 기존 구조에서 바로 실행하는 경우를 위한 fallback
if not MODEL_PATH.exists():
    MODEL_PATH = Path("walk_route_model.json")

if not TRAINING_DATA_PATH.exists():
    TRAINING_DATA_PATH = Path("route_training_data.csv")

print("MODEL_PATH:", MODEL_PATH)
print("TRAINING_DATA_PATH:", TRAINING_DATA_PATH)

MODEL_PATH: walk_route_model.json
TRAINING_DATA_PATH: route_training_data.csv


# 3. 산책 후보 경로 생성 로직

## 3-1. 방향(Bearing) 계산

현재 위치에서 후보 노드가 어느 방향에 있는지 계산한다.
여러 방향의 경유지를 선택하여 출발점으로 돌아오는 순환형 산책 경로를 만들 때 사용한다.

In [3]:
def calculate_bearing(lat1, lon1, lat2, lon2):
    lat1 = math.radians(lat1)
    lat2 = math.radians(lat2)

    diff_lon = math.radians(lon2 - lon1)

    x = math.sin(diff_lon) * math.cos(lat2)
    y = (
        math.cos(lat1) * math.sin(lat2)
        - math.sin(lat1) * math.cos(lat2) * math.cos(diff_lon)
    )

    bearing = math.degrees(math.atan2(x, y))
    return (bearing + 360) % 360

## 3-2. 현재 위치 주변 보행 그래프 생성

OSMnx의 `graph_from_point()`를 사용하여 현재 위치 주변의 보행 가능한 도로 그래프를 생성한다.
출발점은 현재 좌표와 가장 가까운 OSM 노드로 지정한다.

In [4]:
def create_walk_graph(latitude, longitude, dist=1000):
    G = ox.graph_from_point(
        (latitude, longitude),
        dist=dist,
        network_type="walk"
    )

    start_node = ox.distance.nearest_nodes(
        G,
        X=longitude,
        Y=latitude
    )

    return G, start_node

## 3-3. 경유지 후보 노드 선택

현재 위치에서 일정 반경 안에 있는 OSM 노드를 경유지 후보로 수집한다.

In [5]:
def get_candidate_nodes(
    G,
    latitude,
    longitude,
    min_radius,
    max_radius
):
    candidate_nodes = []

    for node, data in G.nodes(data=True):
        distance = ox.distance.great_circle(
            latitude,
            longitude,
            data["y"],
            data["x"]
        )

        if min_radius <= distance <= max_radius:
            candidate_nodes.append(node)

    return candidate_nodes

## 3-4. 방향별 경유지 선택

0~360도 방향을 기준으로 목표 방향과 가장 가까운 후보 노드를 경유지로 선택한다.

In [6]:
def select_waypoints(
    G,
    candidate_nodes,
    target_bearings,
    latitude,
    longitude
):
    if not candidate_nodes:
        return []

    waypoints = []

    for target_bearing in target_bearings:
        best_node = None
        smallest_diff = float("inf")

        for node in candidate_nodes:
            node_data = G.nodes[node]

            bearing = calculate_bearing(
                latitude,
                longitude,
                node_data["y"],
                node_data["x"]
            )

            diff = abs(
                (bearing - target_bearing + 180) % 360 - 180
            )

            if diff < smallest_diff:
                smallest_diff = diff
                best_node = node

        if best_node is not None:
            waypoints.append(best_node)

    return waypoints

## 3-5. 순환형 산책 경로 생성

`출발점 → 경유지 1 → 경유지 2 → 경유지 3 → 출발점` 순서로 최단 경로를 연결한다.

In [7]:
def generate_loop_route(
    G,
    start_node,
    waypoints
):
    route_nodes = [start_node] + waypoints + [start_node]
    loop_route = []

    for i in range(len(route_nodes) - 1):
        section = nx.shortest_path(
            G,
            source=route_nodes[i],
            target=route_nodes[i + 1],
            weight="length"
        )

        if i == 0:
            loop_route.extend(section)
        else:
            loop_route.extend(section[1:])

    return loop_route

## 3-6. 거리 및 경로 중복률 분석

동일한 도로 구간을 다시 이용한 거리를 계산하여 `overlap_ratio`로 사용한다.
반대 방향으로 이동해도 동일한 도로 구간으로 간주한다.

In [8]:
def analyze_route(G, route):
    total_distance = 0
    edge_distances = defaultdict(list)

    for u, v in zip(route[:-1], route[1:]):
        edge_data = G.get_edge_data(u, v)

        if not edge_data:
            continue

        min_length = min(
            data["length"]
            for data in edge_data.values()
        )

        total_distance += min_length

        edge_key = tuple(sorted((u, v)))
        edge_distances[edge_key].append(min_length)

    duplicate_distance = 0

    for distances in edge_distances.values():
        if len(distances) > 1:
            duplicate_distance += sum(distances[1:])

    overlap_ratio = (
        duplicate_distance / total_distance * 100
        if total_distance > 0
        else 0
    )

    return {
        "distance_m": total_distance,
        "duplicate_distance_m": duplicate_distance,
        "overlap_ratio": overlap_ratio
    }

## 3-7. 도로 유형 비율 계산

경로에 포함된 OSM `highway` 유형을 거리 기준으로 집계한다.
보행로, 주거도로, 주요도로 비율을 XGBoost Feature로 사용한다.

In [9]:
def calculate_highway_ratios(G, route):
    highway_distances = {}
    total_distance = 0

    for u, v in zip(route[:-1], route[1:]):
        edge_data = G.get_edge_data(u, v)

        if not edge_data:
            continue

        edge = min(
            edge_data.values(),
            key=lambda data: data["length"]
        )

        highway = edge.get("highway", "unknown")
        length = edge["length"]

        if isinstance(highway, list):
            highway = highway[0]

        highway_distances[highway] = (
            highway_distances.get(highway, 0) + length
        )

        total_distance += length

    if total_distance == 0:
        return {}

    return {
        highway: distance / total_distance
        for highway, distance in highway_distances.items()
    }

## 3-8. 후보 경로 여러 개 생성

방향 조합과 경유지 반경을 변경하여 여러 개의 순환형 후보 경로를 만들고,
총 거리가 2~3km인 경로만 유지한다.

In [10]:
def generate_route_candidates(latitude, longitude):
    G, start_node = create_walk_graph(
        latitude,
        longitude
    )

    bearing_sets = []

    for start_angle in range(0, 120, 10):
        bearing_sets.append([
            start_angle,
            (start_angle + 120) % 360,
            (start_angle + 240) % 360
        ])

    radius_ranges = [
        (250, 400),
        (300, 450),
        (300, 500),
        (350, 500),
        (350, 550)
    ]

    candidates = []

    for min_radius, max_radius in radius_ranges:
        candidate_nodes = get_candidate_nodes(
            G,
            latitude,
            longitude,
            min_radius,
            max_radius
        )

        if not candidate_nodes:
            continue

        for bearings in bearing_sets:
            try:
                waypoints = select_waypoints(
                    G,
                    candidate_nodes,
                    bearings,
                    latitude,
                    longitude
                )

                if len(waypoints) != len(bearings):
                    continue

                route = generate_loop_route(
                    G,
                    start_node,
                    waypoints
                )

                analysis = analyze_route(
                    G,
                    route
                )

                if 2000 <= analysis["distance_m"] <= 3000:
                    candidates.append({
                        "radius_range": (min_radius, max_radius),
                        "bearings": bearings,
                        "route": route,
                        "distance_m": analysis["distance_m"],
                        "duplicate_distance_m":
                            analysis["duplicate_distance_m"],
                        "overlap_ratio":
                            analysis["overlap_ratio"]
                    })

            except (nx.NetworkXNoPath, nx.NodeNotFound):
                continue

    return G, candidates

# 4. 중복/유사 경로 제거

동일한 경로 또는 대부분 같은 도로를 공유하는 경로를 제거하여
추천 후보가 서로 너무 비슷해지는 것을 방지한다.

In [11]:
def remove_exact_duplicates(candidates):
    unique_candidates = []
    seen_routes = set()

    for candidate in candidates:
        route_key = tuple(candidate["route"])

        if route_key not in seen_routes:
            seen_routes.add(route_key)
            unique_candidates.append(candidate)

    return unique_candidates


def route_similarity(route_a, route_b):
    edges_a = {
        tuple(sorted((u, v)))
        for u, v in zip(route_a[:-1], route_a[1:])
    }

    edges_b = {
        tuple(sorted((u, v)))
        for u, v in zip(route_b[:-1], route_b[1:])
    }

    intersection = edges_a & edges_b
    union = edges_a | edges_b

    if not union:
        return 0.0

    return len(intersection) / len(union)


def remove_similar_routes(
    candidates,
    similarity_threshold=0.8
):
    deduplicated_candidates = []

    for candidate in candidates:
        is_similar = False

        for saved_candidate in deduplicated_candidates:
            similarity = route_similarity(
                candidate["route"],
                saved_candidate["route"]
            )

            if similarity >= similarity_threshold:
                is_similar = True
                break

        if not is_similar:
            deduplicated_candidates.append(candidate)

    return deduplicated_candidates

# 5. 공원/녹지 Feature

OSM에서 공원, 정원, 잔디, 숲, 휴양지 등의 공간 데이터를 조회한다.

현재 `green_ratio`는 경로 길이 자체가 아니라 **경로 노드 중 녹지 50m 이내에 위치한 노드의 비율**이다.
프로토타입용 근사치이며 추후에는 선형 경로 길이 기준으로 개선할 수 있다.

In [12]:
def get_green_areas(latitude, longitude, dist=1000):
    tags = {
        "leisure": ["park", "garden"],
        "landuse": ["grass", "forest", "recreation_ground"],
        "natural": ["wood"]
    }

    green_areas = ox.features_from_point(
        (latitude, longitude),
        tags=tags,
        dist=dist
    )

    return green_areas


def calculate_green_ratio(
    G,
    route,
    green_areas,
    buffer_m=50
):
    if green_areas.empty:
        return 0.0

    route_nodes = G.subgraph(route)

    route_gdf = ox.graph_to_gdfs(
        route_nodes,
        nodes=True,
        edges=False
    )

    route_gdf = route_gdf.to_crs(epsg=5179)
    green_gdf = green_areas.to_crs(epsg=5179)

    green_buffer = (
        green_gdf.geometry
        .buffer(buffer_m)
        .union_all()
    )

    near_green = route_gdf.geometry.intersects(
        green_buffer
    )

    if len(route_gdf) == 0:
        return 0.0

    return float(
        near_green.sum() / len(route_gdf)
    )

# 6. 경로 Feature 추출

각 후보 경로에서 XGBoost 입력 Feature 6개를 계산한다.

In [13]:
def extract_route_features(
    G,
    route,
    green_areas
):
    analysis = analyze_route(G, route)
    ratios = calculate_highway_ratios(G, route)

    walkway_ratio = (
        ratios.get("footway", 0)
        + ratios.get("path", 0)
        + ratios.get("pedestrian", 0)
        + ratios.get("living_street", 0)
    )

    residential_ratio = ratios.get(
        "residential",
        0
    )

    major_road_ratio = (
        ratios.get("primary", 0)
        + ratios.get("primary_link", 0)
        + ratios.get("secondary", 0)
        + ratios.get("secondary_link", 0)
    )

    green_ratio = calculate_green_ratio(
        G,
        route,
        green_areas
    )

    return {
        "distance_m": float(
            analysis["distance_m"]
        ),
        "overlap_ratio": float(
            analysis["overlap_ratio"]
        ),
        "walkway_ratio": float(
            walkway_ratio
        ),
        "residential_ratio": float(
            residential_ratio
        ),
        "major_road_ratio": float(
            major_road_ratio
        ),
        "green_ratio": float(
            green_ratio
        )
    }

# 7. 현재 좌표 → 후보 데이터셋 생성

후보 생성, 중복 제거, 녹지 조회, Feature 추출을 하나의 함수로 묶는다.
이 함수는 실시간 추천 API에서도 그대로 사용한다.

In [14]:
def create_route_dataframe(
    G,
    candidates,
    green_areas
):
    columns = [
        "candidate_id",
        "distance_m",
        "overlap_ratio",
        "walkway_ratio",
        "residential_ratio",
        "major_road_ratio",
        "green_ratio",
        "radius_min",
        "radius_max"
    ]

    if not candidates:
        return pd.DataFrame(columns=columns)

    dataset = []

    for i, candidate in enumerate(
        candidates,
        start=1
    ):
        features = extract_route_features(
            G,
            candidate["route"],
            green_areas
        )

        features["candidate_id"] = i
        features["radius_min"] = (
            candidate["radius_range"][0]
        )
        features["radius_max"] = (
            candidate["radius_range"][1]
        )

        dataset.append(features)

    return pd.DataFrame(dataset)[columns]


def generate_route_dataset(
    latitude,
    longitude
):
    G, candidates = generate_route_candidates(
        latitude,
        longitude
    )

    unique_candidates = remove_exact_duplicates(
        candidates
    )

    deduplicated_candidates = remove_similar_routes(
        unique_candidates,
        similarity_threshold=0.8
    )

    green_areas = get_green_areas(
        latitude,
        longitude
    )

    df = create_route_dataframe(
        G,
        deduplicated_candidates,
        green_areas
    )

    df["start_latitude"] = latitude
    df["start_longitude"] = longitude

    return G, deduplicated_candidates, df

# 8. XGBoost 모델 학습 / 재학습

## 학습 데이터

기존 프로젝트에서는 총 96개의 후보 경로를 생성하고 그중 20개에 `walk_score(1~5)`를 수동 라벨링하여
XGBoost 회귀 모델의 프로토타입을 학습했다.

서비스에 사용할 최종 모델은 테스트용 80/20 분리 모델이 아니라,
**성능 확인 후 라벨링된 전체 데이터로 다시 학습한 모델**을 저장하는 것이 적절하다.

아래 셀은 `route_training_data.csv`가 존재할 때 실행한다.

In [15]:
if TRAINING_DATA_PATH.exists():
    training_df = pd.read_csv(
        TRAINING_DATA_PATH
    )

    labeled_df = training_df[
        training_df["walk_score"].notna()
    ].copy()

    X = labeled_df[FEATURE_COLUMNS]
    y = labeled_df["walk_score"].astype(float)

    print("라벨링 데이터:", len(labeled_df))
    print("X:", X.shape)
    print("y:", y.shape)
else:
    print(
        "route_training_data.csv가 없습니다. "
        "기존 노트북의 all_routes_df를 CSV로 저장한 뒤 실행하세요."
    )

route_training_data.csv가 없습니다. 기존 노트북의 all_routes_df를 CSV로 저장한 뒤 실행하세요.


## 8-1. 테스트용 학습 및 MAE 확인

데이터가 적기 때문에 이 수치는 모델 품질을 확정하는 성능 지표라기보다
학습/예측 파이프라인이 동작하는지 확인하는 프로토타입 지표로 사용한다.

In [16]:
if TRAINING_DATA_PATH.exists():
    X_train, X_test, y_train, y_test = (
        train_test_split(
            X,
            y,
            test_size=0.2,
            random_state=42
        )
    )

    test_model = XGBRegressor(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        random_state=42
    )

    test_model.fit(
        X_train,
        y_train
    )

    predictions = test_model.predict(
        X_test
    )

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    result_df = X_test.copy()
    result_df["actual_score"] = y_test
    result_df["predicted_score"] = predictions

    print("MAE:", round(mae, 3))
    display(
        result_df[
            ["actual_score", "predicted_score"]
        ]
    )

## 8-2. 전체 라벨 데이터로 최종 모델 학습

테스트가 끝난 뒤 라벨링된 전체 데이터를 다시 사용하여 서비스용 모델을 생성한다.

In [17]:
if TRAINING_DATA_PATH.exists():
    final_model = XGBRegressor(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        random_state=42
    )

    final_model.fit(X, y)

    MODEL_PATH.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    final_model.save_model(
        MODEL_PATH
    )

    print(
        "최종 모델 저장 완료:",
        MODEL_PATH
    )

# 9. 서비스용 모델 로드

실제 FastAPI 서버는 학습을 매 요청마다 다시 수행하지 않는다.
서버 시작 시 저장된 `walk_route_model.json`을 한 번 로드하고,
새로운 현재 위치에서 생성된 후보 경로에 대해서만 추론한다.

In [18]:
def load_walk_model(
    model_path=MODEL_PATH
):
    model = XGBRegressor()
    model.load_model(model_path)
    return model


if MODEL_PATH.exists():
    walk_model = load_walk_model()
    print("모델 로드 완료:", MODEL_PATH)
else:
    walk_model = None
    print(
        "walk_route_model.json이 없습니다. "
        "모델 파일 경로를 확인하세요."
    )

모델 로드 완료: walk_route_model.json


# 10. OSM 경로 Node ID → 지도 좌표 변환

OSMnx의 경로는 Node ID 배열로 구성된다.

React/Kakao Map에서는 위도/경도 좌표 목록이 필요하므로,
각 노드를 `{latitude, longitude}` 형태로 변환한다.

In [19]:
def route_to_coordinates(G, route):
    coordinates = []

    for sequence, node in enumerate(
        route,
        start=1
    ):
        node_data = G.nodes[node]

        coordinates.append({
            "sequence": sequence,
            "latitude": float(
                node_data["y"]
            ),
            "longitude": float(
                node_data["x"]
            )
        })

    return coordinates

# 11. 실시간 산책로 추천 함수

이 함수가 실제 기능의 핵심이다.

입력:
- 현재 위도
- 현재 경도
- 추천 개수(`top_k`, 기본 4)

처리:
1. OSMnx로 주변 보행 그래프 생성
2. 2~3km 후보 경로 생성
3. Feature 추출
4. XGBoost로 산책 적합도 점수 예측
5. 점수 높은 순으로 정렬
6. 상위 3~4개 경로 반환
7. Kakao Map에 사용할 경로 좌표 포함

In [20]:
def recommend_routes(
    latitude,
    longitude,
    model,
    top_k=4
):
    G, candidates, route_df = (
        generate_route_dataset(
            latitude,
            longitude
        )
    )

    if route_df.empty:
        return []

    X_input = route_df[FEATURE_COLUMNS]

    predictions = model.predict(
        X_input
    )

    route_df = route_df.copy()

    route_df["predicted_score"] = (
        np.clip(
            predictions,
            1.0,
            5.0
        )
    )

    top_routes = (
        route_df
        .sort_values(
            "predicted_score",
            ascending=False
        )
        .head(top_k)
    )

    results = []

    for rank, (_, row) in enumerate(
        top_routes.iterrows(),
        start=1
    ):
        candidate_index = (
            int(row["candidate_id"]) - 1
        )

        candidate = candidates[
            candidate_index
        ]

        coordinates = route_to_coordinates(
            G,
            candidate["route"]
        )

        distance_m = int(
            round(
                float(row["distance_m"])
            )
        )

        # 약 4km/h 보행 속도 기준 단순 추정
        estimated_minutes = int(
            round(distance_m / 66.7)
        )

        results.append({
            "rank": rank,
            "candidate_id": int(
                row["candidate_id"]
            ),
            "distance_m": distance_m,
            "estimated_minutes":
                estimated_minutes,
            "score": round(
                float(
                    row["predicted_score"]
                ),
                2
            ),
            "overlap_ratio": round(
                float(
                    row["overlap_ratio"]
                ),
                2
            ),
            "walkway_ratio": round(
                float(
                    row["walkway_ratio"]
                ),
                3
            ),
            "residential_ratio": round(
                float(
                    row["residential_ratio"]
                ),
                3
            ),
            "major_road_ratio": round(
                float(
                    row["major_road_ratio"]
                ),
                3
            ),
            "green_ratio": round(
                float(
                    row["green_ratio"]
                ),
                3
            ),
            "points": coordinates
        })

    return results

# 12. 현재 좌표로 실시간 추천 테스트

실제 서비스에서는 React의 `navigator.geolocation`에서 받은 좌표가 FastAPI로 전달된다.
노트북에서는 테스트 좌표를 직접 넣어 전체 파이프라인을 검증한다.

> OSM/Overpass 서버 상태에 따라 그래프 또는 녹지 조회에 시간이 걸리거나 타임아웃이 발생할 수 있다.

In [21]:
# 예시 좌표: 테스트 시 원하는 현재 위치 좌표로 변경
test_latitude = 37.5665
test_longitude = 126.9780

if walk_model is not None:
    recommendations = recommend_routes(
        test_latitude,
        test_longitude,
        model=walk_model,
        top_k=4
    )

    print(
        "추천 경로:",
        len(recommendations)
    )

    for route in recommendations:
        print(
            f"추천 {route['rank']} | "
            f"거리 {route['distance_m']}m | "
            f"예상 {route['estimated_minutes']}분 | "
            f"점수 {route['score']} | "
            f"좌표 {len(route['points'])}개"
        )

추천 경로: 4
추천 1 | 거리 2597m | 예상 39분 | 점수 3.97 | 좌표 54개
추천 2 | 거리 2508m | 예상 38분 | 점수 3.97 | 좌표 48개
추천 3 | 거리 2675m | 예상 40분 | 점수 3.96 | 좌표 65개
추천 4 | 거리 2731m | 예상 41분 | 점수 3.96 | 좌표 62개


# 13. FastAPI 적용 구조

노트북에서 위 추천 함수가 정상 동작하는 것을 확인한 뒤,
실제 서버에서는 아래처럼 분리한다.

```text
ai-server/
├── app/
│   ├── __init__.py
│   ├── main.py
│   └── route_service.py
├── model/
│   └── walk_route_model.json
├── data/
│   └── route_training_data.csv
├── notebooks/
│   └── route_generation_cleaned.ipynb
└── requirements.txt
```

### `route_service.py`

다음 기능을 이동한다.

- `calculate_bearing`
- `create_walk_graph`
- `get_candidate_nodes`
- `select_waypoints`
- `generate_loop_route`
- `analyze_route`
- `calculate_highway_ratios`
- `generate_route_candidates`
- `remove_exact_duplicates`
- `route_similarity`
- `remove_similar_routes`
- `get_green_areas`
- `calculate_green_ratio`
- `extract_route_features`
- `create_route_dataframe`
- `generate_route_dataset`
- `route_to_coordinates`
- `recommend_routes`
- 모델 로드 코드

### `main.py`

FastAPI는 HTTP 요청/응답만 담당하도록 얇게 유지한다.

In [22]:
# 이 셀은 구조 참고용이다.
# 실제 서비스에서는 ai-server/app/main.py 파일로 옮긴다.

FASTAPI_MAIN_EXAMPLE = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

from app.route_service import recommend_routes, walk_model

app = FastAPI()


class RecommendRequest(BaseModel):
    latitude: float
    longitude: float


@app.post("/api/routes/recommend")
def recommend_route_api(
    request: RecommendRequest
):
    try:
        routes = recommend_routes(
            request.latitude,
            request.longitude,
            model=walk_model,
            top_k=4
        )

        return {
            "latitude": request.latitude,
            "longitude": request.longitude,
            "count": len(routes),
            "routes": routes
        }

    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=str(e)
        )
'''

print(FASTAPI_MAIN_EXAMPLE)


from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

from app.route_service import recommend_routes, walk_model

app = FastAPI()


class RecommendRequest(BaseModel):
    latitude: float
    longitude: float


@app.post("/api/routes/recommend")
def recommend_route_api(
    request: RecommendRequest
):
    try:
        routes = recommend_routes(
            request.latitude,
            request.longitude,
            model=walk_model,
            top_k=4
        )

        return {
            "latitude": request.latitude,
            "longitude": request.longitude,
            "count": len(routes),
            "routes": routes
        }

    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=str(e)
        )



# 14. 프론트엔드 연동 시 최종 흐름

메인 페이지 진입 시:

1. 브라우저 `navigator.geolocation.getCurrentPosition()`으로 현재 위치 획득
2. FastAPI `/api/routes/recommend`에 위도/경도 전달
3. FastAPI에서 OSMnx 후보 경로 실시간 생성
4. XGBoost로 후보별 산책 적합도 점수 예측
5. 상위 4개 경로와 GPS 좌표 배열 반환
6. 추천 산책로 카드 목록 출력
7. 사용자가 경로를 선택하면 Kakao Map Polyline으로 `points` 표시

### API 요청 예시

```json
{
  "latitude": 37.5665,
  "longitude": 126.9780
}
```

### 응답 구조 예시

```json
{
  "latitude": 37.5665,
  "longitude": 126.9780,
  "count": 4,
  "routes": [
    {
      "rank": 1,
      "candidate_id": 5,
      "distance_m": 2632,
      "estimated_minutes": 39,
      "score": 4.81,
      "overlap_ratio": 19.7,
      "walkway_ratio": 0.768,
      "residential_ratio": 0.0,
      "major_road_ratio": 0.0,
      "green_ratio": 1.0,
      "points": [
        {
          "sequence": 1,
          "latitude": 37.5665,
          "longitude": 126.9780
        }
      ]
    }
  ]
}
```

## 다음 구현 단계

이 노트북에서 `recommend_routes()`가 실제 좌표로 정상 동작하는 것을 확인한 다음,
`route_service.py`와 `main.py`를 생성하여 FastAPI 서버로 분리한다.

In [23]:
test_latitude = 37.621353174310144
test_longitude = 126.92621750262337

recommendations = recommend_routes(
    test_latitude,
    test_longitude,
    model=walk_model,
    top_k=4
)

print("추천 경로 개수:", len(recommendations))

for route in recommendations:
    print(
        f"추천 {route['rank']} | "
        f"거리 {route['distance_m']}m | "
        f"예상 {route['estimated_minutes']}분 | "
        f"점수 {route['score']} | "
        f"좌표 {len(route['points'])}개"
    )

추천 경로 개수: 4
추천 1 | 거리 2649m | 예상 40분 | 점수 4.54 | 좌표 73개
추천 2 | 거리 2613m | 예상 39분 | 점수 4.54 | 좌표 70개
추천 3 | 거리 2610m | 예상 39분 | 점수 4.54 | 좌표 72개
추천 4 | 거리 2652m | 예상 40분 | 점수 4.54 | 좌표 57개
